# FX10 images not in the same format as FX17? FML.

- figuring out how they are presented. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from matplotlib import patches
import matplotlib
from matplotlib.figure import Figure
from numpy.typing import NDArray
from collections.abc import Iterable
import h5py

In [ ]:

def plot_stripes_fx10(path: str, x_white: int, y_white: int, h_white: int,
                 alignment: str = 'N', vstep: int = 0) -> Figure:
    """Plot the image strips and visualize the white reference areas.

    Sometimes the easel/painting/camera is ever so slightly crooked,
    and one spot to clip white reference will not work with all strips.

    Args:
        path (str): Path to the HDF5 file containing the `img_data_*` strips.
        x_white (int): Left edge (x) of the white reference rectangle.
        y_white (int): Top edge (y) of the white reference rectangle on the first strip.
        h_white (int): Height of the white reference rectangle.
        alignment (str, optional): 'N' (top) or 'S' (bottom) anchor. Defaults to 'N'.
        vstep (int, optional): Each successive strip's rectangle is lifted by this
            many pixels more than the previous one. Defaults to 0.

    Raises:
        ValueError: If alignment is not 'N' or 'S'.

    Returns:
        Figure: The matplotlib figure with all strips.
    """
    if alignment not in ('N', 'S'):
        raise ValueError("alignment must be either 'N' (top/north) or 'S' (bottom/south).")

    with h5py.File(path, "r") as f:
        keys = sorted(
            (k for k in f.keys() if k.startswith("img_data_") or k.startswith('data')),
            key=lambda k: int(k.split('_')[-1]),
        )
        cubes: list[NDArray] = [f[k][:] for k in keys]  # read everything in one open  # type: ignore

    cols = len(keys)
    fig, axes = plt.subplots(1, cols, dpi=100, squeeze=False)
    axes = axes[0]  # squeeze=False keeps this indexable even for a single strip

    for idx, cube in enumerate(cubes):
        # Camera mounted upside down --> image upside down and white strip on top if the strips
        y = y_white + idx * vstep

        rect = patches.Rectangle(
            (x_white, y),
            width=cube.shape[1] - x_white,
            height=h_white,
            linewidth=1,
            edgecolor='red',
            facecolor='none',
        )

        axes[idx].imshow(cube[:, :, 100])
        axes[idx].set_anchor(alignment)
        axes[idx].axis('off')
        axes[idx].add_patch(rect)

    fig.tight_layout()
    return fig